# 22DM015 Final Project — Financial PhraseBank
## Person B: Part 3 — Full-dataset SOA comparison

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453); `labeled_32.csv` is loaded separately.‍
- Evaluate headline numbers on **`test`** only.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍
- Part 2 (BERT fine-tuning + augmentation) lives in `notebooks/bert_part2.ipynb`; this notebook reads its rows from the shared scoreboard to overlay them on the curve.‍

### What this notebook does
- **3a.** Train on 1 / 10 / 25 / 50 / 75 / 100 % of `train` (use `du.subset_by_fraction`).‍
- **3b.** Plot the learning curve.‍
- **3c.** Fold in Part 2 techniques (read from `results/results.csv`).‍
- **3d.** Methodology analysis (student-authored).‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [13]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
import pandas as pd
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical train/val/test for everyone
train, val, test = splits['train'], splits['val'], splits['test']
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

> **Install (run once):** `transformers`, `torch`, `accelerate` are needed here.‍ On Python 3.14 torch wheels may be missing — use a 3.11/3.12 venv.‍

## Part 3 setup — shared training helpers
Same training protocol as `bert_part2.ipynb` so the curve points are directly comparable to the Part 2 rows that get overlaid in 3c.‍ The helpers (`train_bert`, `eval_split`, `logged`, `notes_kv`, `fmt`) are duplicated here on purpose: the two notebooks remain runnable independently.‍

In [14]:
# watermark: AGLLM (AI-assisted content disclosure)
# Shared training helpers (mirrors bert_part2.ipynb so Part 3 stays runnable on its own).
import logging, warnings
# Tell transformers to skip its TensorFlow/Flax/Keras probe (we use PyTorch only).
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_ADVISORY_WARNINGS', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
logging.getLogger('tensorflow').setLevel(logging.ERROR)
logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
logging.getLogger('torchao').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message=r'.*Skipping import of cpp extensions.*')

import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments, set_seed)

torch.set_num_threads(os.cpu_count() or 4)   # all trainings here are CPU-bound
MODEL = 'bert-base-uncased'      # same as Part 2 for direct comparability
NUM_LABELS = 3
MAX_LEN = 128

tok = AutoTokenizer.from_pretrained(MODEL)


def encode(df, max_len=MAX_LEN):
    ds = Dataset.from_pandas(df[['text', 'label']], preserve_index=False)
    return ds.map(lambda b: tok(b['text'], truncation=True, padding='max_length', max_length=max_len),
                  batched=True)


def train_bert(train_df, out_dir, *, epochs=20, batch=8, max_len=MAX_LEN, log_epochs=False):
    """One shared fine-tuning protocol; Part 3 curve uses 64 / 16 / 3 (see CURVE below)."""
    set_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=NUM_LABELS)
    args = TrainingArguments(
        output_dir=out_dir, seed=SEED,
        num_train_epochs=epochs, per_device_train_batch_size=batch,
        per_device_eval_batch_size=64, learning_rate=2e-5,
        eval_strategy='no', save_strategy='no',
        logging_strategy='epoch' if log_epochs else 'no',
        report_to='none', disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=encode(train_df, max_len))
    trainer.train()
    trainer.eval_max_len = max_len
    return trainer


def eval_split(trainer, df, max_len=None):
    max_len = max_len or getattr(trainer, 'eval_max_len', MAX_LEN)
    pred = trainer.predict(encode(df, max_len)).predictions.argmax(-1)
    return eu.evaluate(df['label'].values, pred)


def logged(method, full_row=False):
    """Latest TEST row for (MODEL, method) from the shared scoreboard. The eval module
    keys rows on (model, method, split, n_train_labeled) and no longer tracks person, so
    we match on MODEL + method here (each method has a single n in this notebook).
    Delete a row from results.csv to force that experiment to re-run."""
    if not eu.RESULTS_CSV.exists():
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['model'] == MODEL) & (r['method'] == method) & (r['split'] == 'test')]
    if not len(r):
        return None
    row = r.iloc[-1]
    return row if full_row else {k: row[k] for k in eu.METRIC_KEYS}


def notes_kv(notes):
    """Parse the 'k=v; k=v' segments of a notes string into a dict."""
    out = {}
    for seg in str(notes).split(';'):
        if '=' in seg:
            k, v = seg.split('=', 1)
            out[k.strip()] = v.strip()
    return out


fmt = eu.fmt

## Part 3a.‍ Data-scaling curve (1 / 10 / 25 / 50 / 75 / 100 %)

In [15]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3a: data-scaling curve — fine-tune a fresh BERT on increasing fractions of train,
# evaluate on test. Uses the shared train_bert/eval_split with a lighter fixed protocol
# (max_len 64, batch 16, 3 epochs) so the curve isolates data quantity.
# RESUME-AWARE + PROTOCOL-SAFE: a logged fraction is reused only if its notes match the
# current CURVE protocol (epochs/maxlen) — editing CURVE retrains instead of silently
# mixing protocols. Six bert-base trainings take ~1.5 h on CPU; delete rows to force.
# NOTE: du.subset_by_fraction samples each fraction independently (stratified, same
# seed) — fractions are NOT nested subsets of each other; see its docstring.
CURVE = dict(epochs=3, batch=16, max_len=64)
FRACTIONS = [0.01, 0.10, 0.25, 0.50, 0.75, 1.00]

curve_rows = []
for f in FRACTIONS:
    method = f'full-{int(f * 100)}%'
    sub = du.subset_by_fraction(train, f)
    prev = logged(method, full_row=True)
    m = None
    if prev is not None:
        kv = notes_kv(prev['notes'])
        if kv.get('epochs') == str(CURVE['epochs']) and kv.get('maxlen') == str(CURVE['max_len']):
            m = {k: prev[k] for k in eu.METRIC_KEYS}
            print(f"frac={f:>4}  n={len(sub):4d}  [cached]  macroF1={float(m['f1_macro']):.4f}")
        else:
            print(f"frac={f:>4}  [stale] logged row used a different protocol — retraining")
    if m is None:
        tr = train_bert(sub, '../.cache/bert_curve', **CURVE)
        m = eval_split(tr, test)
        eu.log_result(MODEL, method, len(sub), m,
                      notes=f"frac={f}; epochs={CURVE['epochs']}; maxlen={CURVE['max_len']}")
        print(f"frac={f:>4}  n={len(sub):4d}  [trained] macroF1={float(m['f1_macro']):.4f}")
    curve_rows.append({'frac': f, 'n_train': len(sub), **m})

pd.DataFrame(curve_rows)

## Part 3b / 3c.‍ Learning curve + Part 2 techniques overlay

In [17]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 3b: learning curve from the shared results scoreboard.
# Part 3c: overlay every Part 2 technique logged in results.csv on the same axes, each at
# the number of training rows it used — i.e. "how many real labels is each technique worth?"
# Person C's LLM rows (zero-/few-shot) and Person A's Part 1 baselines are drawn as
# horizontal reference lines (no x-position on the log axis). Baselines fall back to the
# Part 1 notebook's numbers until Person A logs them through eu.log_result.
import matplotlib.pyplot as plt

res = pd.read_csv(eu.RESULTS_CSV)
bsel = (res['model'] == MODEL)
# log_result upserts on (model, method, split, n_train_labeled), so rows are already unique
curve = res[bsel & res['method'].str.startswith('full-')].sort_values('n_train_labeled')

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(curve['n_train_labeled'], curve['f1_macro'], marker='o', label='BERT macro-F1 (data-scaling)')
ax.plot(curve['n_train_labeled'], curve['accuracy'], marker='s', ls='--', label='BERT accuracy')

# Part 2 techniques (3c) — only the ones already logged are drawn, so this cell
# degrades gracefully until 2d/2e have run
TECHNIQUES = [('32-shot', 'red', '2a 32-shot'),
              ('augmented', 'darkorange', '2b augmented (back-translation)'),
              ('llm-generated', 'purple', '2d 32 + LLM-generated'),
              ('optimal-combo', 'saddlebrown', '2e optimal combo')]
tech_rows = []
for method, color, label in TECHNIQUES:
    row = res[bsel & (res['method'] == method)]
    if len(row):
        ax.scatter(row['n_train_labeled'], row['f1_macro'], color=color, zorder=5,
                   label=f'{label} macro-F1')
        tech_rows.append(row)

# Person C's zero-/few-shot LLM rows (Part 2c) as horizontal reference lines
llm = res[res['method'].isin(['zero-shot', 'few-shot'])]
for _, r in llm.iterrows():
    ax.axhline(float(r['f1_macro']), ls='-.', lw=1,
               label=f"{r['model']} {r['method']} (macro-F1 {float(r['f1_macro']):.2f})")

# Part 1 reference floors (test split): read Person A's logged rows when available,
# otherwise fall back to the numbers from the Part 1 notebook
for pattern, fallback, color, label in [('random', 0.3035, 'gray', 'random floor'),
                                        ('rule', 0.7304, 'green', 'rule-based')]:
    row = res[res['method'].str.contains(pattern, case=False, na=False)]
    f1 = float(row['f1_macro'].iloc[-1]) if len(row) else fallback
    ax.axhline(f1, color=color, ls=':', label=f'{label} (macro-F1 {f1:.2f})')

ax.set_xscale('log')
ax.set_xlabel('# training examples used (log scale)')
ax.set_ylabel('test score')
ax.set_title('Part 3 learning curve — BERT on Financial PhraseBank (allagree)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

cols = ['method', 'n_train_labeled', 'accuracy', 'f1_macro', 'f1_weighted']
pd.concat([curve[cols]] + [r[cols] for r in tech_rows] + ([llm[cols]] if len(llm) else []),
          ignore_index=True)

### 3c — technique comparison &nbsp;·&nbsp; 3d — methodology analysis

**3c (fold in Part 2 techniques):** the cell above overlays every Part 2 row logged in `results/results.csv` (`32-shot`, `augmented`, `llm-generated`, `optimal-combo`) on the data-scaling curve, each plotted at the number of training rows it used — i.e.‍ "how many real labels is each technique worth?".‍ Points appear automatically as 2d/2e get logged; re-run the cell after they finish.‍

_TODO (student-authored) — 3d methodology analysis._ Compare all methods (random / rule-based / 32-shot / augmentation / LLM-generated / full-data curve): where each helps, the data-efficiency story, limitations, and what you'd do with more compute.‍ Your words; commit with `--no-verify`.‍